In [ ]:
import itertools
import json
import time
from dataclasses import asdict, replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uavloc.datasets import dataset_ref
from uavloc.filtering.empty_view import EmptyViewModel
from uavloc.plotting.flight import plot_end_error_cdf
from uavloc.simulation.arms import FilterParams, HeadSpec, build_correlator, estimator_factory
from uavloc.simulation.sequence import (all_fixes, flight_metrics, load_flight_set, per_flight_table, run_flights,
                                        village_bootstrap)
from uavloc.track import experiment_dir, load_runs, log_run, set_seed

# Params

In [ ]:
EXP = "032-empty-view-likelihood"
SEED = 42
DEVICE = "cpu"                    # CPU-only process: MPS corrupts CPU conv2d in the same process (bug-121)
ROUTE_SET = "sim-flights"
INITIAL_ERROR_M = {"normal": 0.0, "cold": 40.0}
SIGMA0_M = {"normal": 1.0, "cold": 25.0}
CORRELATOR = {
    "method": "segmentation", "seg_architecture": "lraspp",
    "seg_checkpoint_drone": "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt",
    "seg_checkpoint_sat": "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt",
    "seg_channels": "all", "drone_seg_gsd_m": 0.385, "match_gsd_m": 0.385, "matching": "weighted-ncc",
    "class_weights": [0.0, 1.0, 1.0, 0.5, 0.0, 0.5], "window": "rect", "search_scale": 1.7, "fov_w_deg": 60.0,
    "sat_feature_crop_pct": 0.03, "drone_feature_crop_pct": 0.0,
}
N_PARTICLES = 2000
TEMPERATURE = 0.5                 # 027.12 / 027.4, tuned in 027
Q_POS_M_PER_SQRT_S = 2.0
WINDOW_CAP_HALF_M = 100.0
WINDOW_MARGIN_M = 2.0
HEAD_RUN = "023.3"
HEAD_EXP = "023-learned-correlation-head"
HEAD_ENCODER_CHANNELS = [32, 32, 16]
HEAD_REFINE_CHANNELS = 16
HEAD_CLEAN_VILLAGES = ["bushcha-50p2422-25p8305", "ivannie-50p4543-25p7398", "maidan-50p2512-26p0155",
                       "maksymivka-48p8871-23p9317", "oblaznytsia-49p2703-24p1288", "pidhaichyky-48p5627-25p1788",
                       "pidhorodnie-49p5387-25p5199", "sadzhavka-48p5678-24p7897", "strilche-48p6821-25p5613",
                       "vesnianka-49p7175-27p3183", "zlynets-50p3697-25p7999"]   # villages the 023 head never trained on
MODEL_PATH = "artifacts/empty_view.json"
TUNING_DIR = "artifacts/tuning"
SCENARIO = "normal"
SPLIT = "test"
LIKELIHOOD = "head"               # "head" (032.1) | "ncc" (032.2)
BETA = 1.0                        # from tuning, set per papermill run
LOG_RUN = True
WAIT_FOR_RUNS = 0
REGISTERED_RUN = "032.1"
BASELINE_RUN = "027.12"
BASELINE_EXP = "027-sequence-filters"
WORST_BASELINE_FLIGHTS = 10
BOOTSTRAP_RESAMPLES = 10_000
NOTE = ""

In [ ]:
set_seed(SEED)

# Data

In [ ]:
set_config, specs = load_flight_set(ROUTE_SET, split=SPLIT)
tuning = pd.concat([pd.read_csv(p) for p in sorted(Path(TUNING_DIR).glob("shard-[0-9].csv"))], ignore_index=True)
baseline_dir = experiment_dir(BASELINE_EXP) / "artifacts"
baseline = pd.read_csv(baseline_dir / f"flights-{BASELINE_RUN}.csv")
baseline_fixes = pd.read_csv(baseline_dir / f"fixes-{BASELINE_RUN}.csv")
len(specs), len(tuning), len(baseline)

# Model

In [ ]:
model_record = json.loads(Path(MODEL_PATH).read_text())
empty_view = EmptyViewModel(**model_record["model"])
empty_view

In [ ]:
def filter_params(empty_view: EmptyViewModel | None, beta: float) -> FilterParams:
    return FilterParams(TEMPERATURE, Q_POS_M_PER_SQRT_S, N_PARTICLES, 0, 0, 0.0, 0.0, 0.0, 0.0, WINDOW_CAP_HALF_M,
                        WINDOW_MARGIN_M, empty_view, beta)

In [ ]:
def head_spec() -> HeadSpec:
    return HeadSpec(experiment_dir(HEAD_EXP) / "artifacts" / f"head-{HEAD_RUN}.pt", tuple(HEAD_ENCODER_CHANNELS),
                    HEAD_REFINE_CHANNELS)

In [ ]:
def correlator_for(map_raster):
    return build_correlator(map_raster, CORRELATOR, DEVICE, head_spec() if LIKELIHOOD == "head" else None)

In [ ]:
params = filter_params(empty_view, BETA)
factories = {"particles": estimator_factory("particles", params)}

# Run

In [ ]:
outcomes = run_flights(set_config, specs, INITIAL_ERROR_M[SCENARIO], SIGMA0_M[SCENARIO], correlator_for, factories)

In [ ]:
table = per_flight_table(outcomes["particles"])
fixes = all_fixes(outcomes["particles"])
metrics = flight_metrics(outcomes["particles"])
pd.Series(metrics)

In [ ]:
def paired(a: pd.DataFrame, b: pd.DataFrame, prefix: str, columns: tuple[str, ...]) -> dict[str, float]:
    out = {}
    for column in columns:
        delta, low, high = village_bootstrap(a, b, column, BOOTSTRAP_RESAMPLES, SEED)
        out.update({f"{prefix}_{column}_delta": delta, f"{prefix}_{column}_ci_low": low, f"{prefix}_{column}_ci_high": high})
    return out

In [ ]:
clean = table["village"].isin(HEAD_CLEAN_VILLAGES)
clean_baseline = baseline["village"].isin(HEAD_CLEAN_VILLAGES)
worst = baseline.nlargest(WORST_BASELINE_FLIGHTS, "end_error_m")[["village", "route_index"]]
worst_now = table.merge(worst, on=["village", "route_index"])
comparison = {
    **paired(table, baseline, "vs_baseline", ("end_error_m", "mean_error_m", "diverged")),
    **paired(table[clean], baseline[clean_baseline], "clean_vs_baseline", ("end_error_m", "mean_error_m")),
    "baseline_mean_end_error_m": float(baseline["end_error_m"].mean()),
    "ratio_to_baseline_mean_end_error": metrics["mean_end_error_m"] / float(baseline["end_error_m"].mean()),
    "clean_flights": int(clean.sum()), "clean_mean_end_error_m": float(table.loc[clean, "end_error_m"].mean()),
    "clean_baseline_mean_end_error_m": float(baseline.loc[clean_baseline, "end_error_m"].mean()),
    "empty_view_share": float((fixes["decision"] == "empty-view").mean()),
    "empty_view_fix_error_m": float(fixes.loc[fixes["decision"] == "empty-view", "error_after_m"].mean()),
    "baseline_blind_fix_error_m": float(baseline_fixes.loc[baseline_fixes["decision"] == "blind", "error_after_m"].mean()),
    "worst_baseline_flights_end_error_m": float(worst_now["end_error_m"].mean()),
    "worst_baseline_flights_baseline_end_error_m": float(baseline.nlargest(WORST_BASELINE_FLIGHTS, "end_error_m")["end_error_m"].mean()),
}
comparison

In [ ]:
worst_now.merge(baseline[["village", "route_index", "end_error_m"]], on=["village", "route_index"],
                suffixes=("", "_baseline"))[["village", "route_index", "end_error_m_baseline", "end_error_m",
                                              "skipped_share", "truth_in_window_share"]].round(2)

# Plots

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
plot_end_error_cdf({f"baseline {BASELINE_RUN}": baseline["end_error_m"], f"empty view, β = {BETA}": table["end_error_m"],
                    "INS only": table["end_error_ins_m"]}, ax)
ax.set_title(f"032, particles + {LIKELIHOOD}, {SCENARIO} start, {SPLIT} flights ({len(specs)})")
fig.tight_layout()

# Log

In [ ]:
config = {"registered_run": REGISTERED_RUN, "arm": "particles", "scenario": SCENARIO, "split": SPLIT,
          "route_set": dataset_ref(ROUTE_SET), "initial_error_m": INITIAL_ERROR_M[SCENARIO], "sigma0_m": SIGMA0_M[SCENARIO],
          "ins": asdict(set_config.ins), "correlator": CORRELATOR, "filter": asdict(params),
          "likelihood": f"head {HEAD_RUN}" if LIKELIHOOD == "head" else "weighted-ncc",
          "empty_view_model": model_record, "head_clean_villages": HEAD_CLEAN_VILLAGES,
          "baseline_run": f"{BASELINE_EXP}/{BASELINE_RUN}",
          "tuning": {"grid": {"beta": sorted(tuning["beta"].unique().tolist())}, "likelihood": "head",
                     "chosen_beta": BETA, "tuning_mean_end_error_m": float(tuning.loc[tuning["beta"] == BETA, "mean_end_error_m"].iloc[0])}}

In [ ]:
if LOG_RUN:
    table.to_csv(Path("artifacts") / f"flights-{REGISTERED_RUN}.csv", index=False)
    fixes.to_csv(Path("artifacts") / f"fixes-{REGISTERED_RUN}.csv", index=False)
    while len(load_runs(EXP)) < WAIT_FOR_RUNS:
        time.sleep(30)
    expected = f"032.{len(load_runs(EXP)) + 1}"
    if expected != REGISTERED_RUN:
        raise RuntimeError(f"run would log as {expected}, registered as {REGISTERED_RUN}")
    log_run(exp=EXP, config=config, metrics={**metrics, **comparison},
            note=f"{REGISTERED_RUN}: particles + {LIKELIHOOD} + empty view (β = {BETA}), {SCENARIO} start. {NOTE}",
            figures=[fig], seed=SEED, device=DEVICE)